In [1]:
import pandas as pd
import numpy as np
import joblib
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
model = joblib.load("../models/network_model.joblib")

feature_columns = joblib.load(
    "../models/feature_columns.joblib"
)

print("Model loaded successfully!")
print("Expected features:", len(feature_columns))

Model loaded successfully!
Expected features: 78


In [3]:
# Load live traffic CSV
live_flows = pd.read_csv(
    "../data/processed/live_flows.csv"
)

print("Live traffic loaded successfully!")

print("Live traffic shape:", live_flows.shape)

display(live_flows.head())

Live traffic loaded successfully!
Live traffic shape: (6, 82)


,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,...,bwd_pkts_b_avg,fwd_blk_rate_avg,bwd_blk_rate_avg,fwd_seg_size_avg,bwd_seg_size_avg,cwr_flag_count,subflow_fwd_pkts,subflow_bwd_pkts,subflow_fwd_byts,subflow_bwd_byts
0,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,2603:1063:5:110::365:7ea3,60183,443,6,2026-11-12 16:28:48,0.149431,1579.324237,20.076156,13.384104,...,0,0.000000e+00,0,75.000000,86.000000,0,2,1,150,86
1,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,64:ff9b::3384:c16b,59091,443,6,2026-11-12 16:28:48,0.269672,18552.167077,37.082085,25.957459,...,0,1.933173e+07,0,669.571429,105.333333,0,7,3,4687,316
2,2606:4700:8ca1:d8b7:9a74:ae6:ccf5:ac50,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,443,59636,6,2026-11-12 16:28:48,0.065058,5717.974730,61.483599,46.112699,...,0,0.000000e+00,0,90.000000,102.000000,0,3,1,270,102
3,10.203.56.81,10.203.56.17,57574,53,17,2026-11-12 16:28:48,0.100533,4068.315876,29.840948,19.893965,...,0,0.000000e+00,0,92.000000,225.000000,0,2,1,184,225
4,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,2409:40c1:31d4:32c9::f6,57574,53,17,2026-11-12 16:28:48,0.000000,0.000000,0.000000,0.000000,...,0,0.000000e+00,0,112.000000,0.000000,0,2,0,224,0


In [4]:
feature_mapping = {
    "Destination Port": "dst_port",
    "Flow Duration": "flow_duration",
    "Total Fwd Packets": "tot_fwd_pkts",
    "Total Backward Packets": "tot_bwd_pkts",
    "Total Length of Fwd Packets": "totlen_fwd_pkts",
    "Total Length of Bwd Packets": "totlen_bwd_pkts",
    "Fwd Packet Length Max": "fwd_pkt_len_max",
    "Fwd Packet Length Min": "fwd_pkt_len_min",
    "Fwd Packet Length Mean": "fwd_pkt_len_mean",
    "Fwd Packet Length Std": "fwd_pkt_len_std",
    "Bwd Packet Length Max": "bwd_pkt_len_max",
    "Bwd Packet Length Min": "bwd_pkt_len_min",
    "Bwd Packet Length Mean": "bwd_pkt_len_mean",
    "Bwd Packet Length Std": "bwd_pkt_len_std",
    "Flow Bytes/s": "flow_byts_s",
    "Flow Packets/s": "flow_pkts_s",
    "Flow IAT Mean": "flow_iat_mean",
    "Flow IAT Std": "flow_iat_std",
    "Flow IAT Max": "flow_iat_max",
    "Flow IAT Min": "flow_iat_min",
    "Fwd IAT Total": "fwd_iat_tot",
    "Fwd IAT Mean": "fwd_iat_mean",
    "Fwd IAT Std": "fwd_iat_std",
    "Fwd IAT Max": "fwd_iat_max",
    "Fwd IAT Min": "fwd_iat_min",
    "Bwd IAT Total": "bwd_iat_tot",
    "Bwd IAT Mean": "bwd_iat_mean",
    "Bwd IAT Std": "bwd_iat_std",
    "Bwd IAT Max": "bwd_iat_max",
    "Bwd IAT Min": "bwd_iat_min",
    "Fwd PSH Flags": "fwd_psh_flags",
    "Bwd PSH Flags": "bwd_psh_flags",
    "Fwd URG Flags": "fwd_urg_flags",
    "Bwd URG Flags": "bwd_urg_flags",
    "Fwd Header Length": "fwd_header_len",
    "Bwd Header Length": "bwd_header_len",
    "Fwd Packets/s": "fwd_pkts_s",
    "Bwd Packets/s": "bwd_pkts_s",
    "Min Packet Length": "pkt_len_min",
    "Max Packet Length": "pkt_len_max",
    "Packet Length Mean": "pkt_len_mean",
    "Packet Length Std": "pkt_len_std",
    "Packet Length Variance": "pkt_len_var",
    "FIN Flag Count": "fin_flag_cnt",
    "SYN Flag Count": "syn_flag_cnt",
    "RST Flag Count": "rst_flag_cnt",
    "PSH Flag Count": "psh_flag_cnt",
    "ACK Flag Count": "ack_flag_cnt",
    "URG Flag Count": "urg_flag_cnt",
    "CWE Flag Count": "cwr_flag_count",
    "ECE Flag Count": "ece_flag_cnt",
    "Down/Up Ratio": "down_up_ratio",
    "Average Packet Size": "pkt_size_avg",
    "Avg Fwd Segment Size": "fwd_seg_size_avg",
    "Avg Bwd Segment Size": "bwd_seg_size_avg",
    "Fwd Header Length.1": "fwd_header_len",
    "Fwd Avg Bytes/Bulk": "fwd_byts_b_avg",
    "Fwd Avg Packets/Bulk": "fwd_pkts_b_avg",
    "Fwd Avg Bulk Rate": "fwd_blk_rate_avg",
    "Bwd Avg Bytes/Bulk": "bwd_byts_b_avg",
    "Bwd Avg Packets/Bulk": "bwd_pkts_b_avg",
    "Bwd Avg Bulk Rate": "bwd_blk_rate_avg",
    "Subflow Fwd Packets": "subflow_fwd_pkts",
    "Subflow Fwd Bytes": "subflow_fwd_byts",
    "Subflow Bwd Packets": "subflow_bwd_pkts",
    "Subflow Bwd Bytes": "subflow_bwd_byts",
    "Init_Win_bytes_forward": "init_fwd_win_byts",
    "Init_Win_bytes_backward": "init_bwd_win_byts",
    "act_data_pkt_fwd": "fwd_act_data_pkts",
    "min_seg_size_forward": "fwd_seg_size_min",
    "Active Mean": "active_mean",
    "Active Std": "active_std",
    "Active Max": "active_max",
    "Active Min": "active_min",
    "Idle Mean": "idle_mean",
    "Idle Std": "idle_std",
    "Idle Max": "idle_max",
    "Idle Min": "idle_min"
}

print("Feature mapping defined successfully!")
print("Total mapped features:", len(feature_mapping))

Feature mapping defined successfully!
Total mapped features: 78


In [5]:
print("Total mapped features:", len(feature_mapping))

Total mapped features: 78


In [6]:
missing_features = [
    feature for feature in feature_columns
    if feature not in feature_mapping
]

missing_columns = [
    column for column in feature_mapping.values()
    if column not in live_flows.columns
]

print("Missing model features:", missing_features)
print("Missing live columns:", missing_columns)

Missing model features: []
Missing live columns: []


In [7]:
missing_features = [
    col for col in feature_columns
    if col not in feature_mapping
]

missing_live_columns = [
    col for col in feature_mapping.values()
    if col not in live_flows.columns
]

print("Missing model features:", missing_features)
print("Missing live columns:", missing_live_columns)

Missing model features: []
Missing live columns: []


In [8]:
# Select the 78 required features from live traffic
live_features = live_flows[
    [feature_mapping[col] for col in feature_columns]
].copy()

# Rename columns to match model training feature names
live_features.columns = feature_columns

print("Live features created successfully!")
print("Shape:", live_features.shape)

display(live_features.head())

Live features created successfully!
Shape: (6, 78)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,443,0.149431,2,1,150,86,75,75,75.000000,0.000000,...,2,0,0,0,0,0,0,0,0,0
1,443,0.269672,7,3,4687,316,1374,82,669.571429,611.075131,...,7,0,0,0,0,0,0,0,0,0
2,59636,0.065058,3,1,270,102,98,74,90.000000,11.313708,...,2,0,0,0,0,0,0,0,0,0
3,53,0.100533,2,1,184,225,92,92,92.000000,0.000000,...,2,20,0,0,0,0,0,0,0,0
4,53,0.000000,2,0,224,0,112,112,112.000000,0.000000,...,2,0,0,0,0,0,0,0,0,0


In [9]:
# Check missing values
print("Missing values:", live_features.isnull().sum().sum())

# Check infinite values
print("Infinite values:", np.isinf(live_features.select_dtypes(include=np.number)).sum().sum())

# Check data types
print("\nFeature data types:")
print(live_features.dtypes.value_counts())

# Display statistical summary
display(live_features.describe())

Missing values: 0
Infinite values: 0

Feature data types:
int64      45
float64    33
Name: count, dtype: int64


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,6.000000,6.000000,6.00,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,...,6.000000,6.000000,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0
mean,10178.500000,0.097449,3.00,1.000000,941.166667,121.500000,302.833333,83.500000,184.095238,103.731473,...,2.500000,6.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
std,24229.881087,0.102386,2.00,1.095445,1835.759071,126.259653,525.018254,16.465115,238.356270,248.587813,...,2.345208,10.327956,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
min,53.000000,0.000000,2.00,0.000000,132.000000,0.000000,66.000000,66.000000,66.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,150.500000,0.016265,2.00,0.250000,158.500000,21.500000,79.250000,74.250000,78.750000,0.000000,...,2.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50%,443.000000,0.082795,2.00,1.000000,204.000000,94.000000,95.000000,78.500000,91.000000,0.000000,...,2.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
75%,443.000000,0.137207,2.75,1.000000,258.500000,194.250000,108.500000,89.500000,107.000000,8.485281,...,2.000000,15.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,59636.000000,0.269672,7.00,3.000000,4687.000000,316.000000,1374.000000,112.000000,669.571429,611.075131,...,7.000000,20.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
# Load training data sample
X_train_sample = joblib.load("../models/x_train_sample.joblib")

# Select important features for comparison
selected_features = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Total Fwd Packets",
    "Total Backward Packets"
]

print("Training Data Statistics:")
display(X_train_sample[selected_features].describe())

print("\nLive Traffic Statistics:")
display(live_features[selected_features].describe())

Training Data Statistics:


,Flow Duration,Flow Bytes/s,Flow Packets/s,Total Fwd Packets,Total Backward Packets
count,5.000000e+02,5.000000e+02,5.000000e+02,500.000000,500.000000
mean,1.493631e+07,9.222813e+05,6.824627e+04,4.828000,4.028000
std,3.328229e+07,4.857413e+06,2.560665e+05,9.058174,8.806266
min,1.000000e+00,0.000000e+00,3.121282e-02,1.000000,0.000000
25%,1.347500e+02,6.913124e+01,3.184201e+00,1.000000,1.000000
50%,3.092150e+04,4.029534e+03,1.253070e+02,2.000000,2.000000
75%,2.728500e+06,1.693873e+05,2.531646e+04,4.000000,3.000000
max,1.199312e+08,5.610000e+07,2.000000e+06,104.000000,109.000000



Live Traffic Statistics:


,Flow Duration,Flow Bytes/s,Flow Packets/s,Total Fwd Packets,Total Backward Packets
count,6.000000,6.000000,6.000000,6.00,6.000000
mean,0.097449,4986.296987,24.747131,3.00,1.000000
std,0.102386,7024.879380,23.555573,2.00,1.095445
min,0.000000,0.000000,0.000000,2.00,0.000000
25%,0.016265,394.831059,5.019039,2.00,0.250000
50%,0.082795,2823.820056,24.958552,2.00,1.000000
75%,0.137207,5305.560017,35.271801,2.75,1.000000
max,0.269672,18552.167077,61.483599,7.00,3.000000


In [11]:
# Check negative values
negative_counts = (live_features < 0).sum()

print("Negative values in live features:")
display(negative_counts[negative_counts > 0])

# Check constant columns
constant_columns = [
    col for col in live_features.columns
    if live_features[col].nunique() <= 1
]

print("\nConstant columns:")
print(constant_columns)

# Check duplicate rows
print("\nDuplicate live feature rows:",
      live_features.duplicated().sum())

Negative values in live features:


Series([], dtype: int64)


Constant columns:
['Flow IAT Min', 'Fwd IAT Min', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Count', 'RST Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']

Duplicate live feature rows: 0


In [12]:
live_flows = pd.read_csv("../data/processed/live_flows.csv")

print("Updated CSV shape:", live_flows.shape)
print("CWR flag values:")
print(live_flows["cwr_flag_count"].value_counts())

Updated CSV shape: (6, 82)
CWR flag values:
cwr_flag_count
0    6
Name: count, dtype: int64


In [13]:
print("Protocol distribution:")

print(
    live_flows["protocol"]
    .value_counts()
    .rename(index={
        6: "TCP",
        17: "UDP"
    })
)

print("\nTotal flows:", len(live_flows))

Protocol distribution:
protocol
TCP    4
UDP    2
Name: count, dtype: int64

Total flows: 6


In [14]:
zero_duration = live_flows[
    live_flows["flow_duration"] == 0
]

print("Zero-duration flows:", len(zero_duration))

display(
    zero_duration[
        ["src_ip", "dst_ip", "protocol", "flow_duration",
         "flow_byts_s", "flow_pkts_s"]
    ]
)

Zero-duration flows: 2


,src_ip,dst_ip,protocol,flow_duration,flow_byts_s,flow_pkts_s
4,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,2409:40c1:31d4:32c9::f6,17,0.0,0.0,0.0
5,10.203.56.81,51.132.193.111,6,0.0,0.0,0.0


In [15]:
from pathlib import Path
import os

print("Notebook is running from:", os.getcwd())
print()

processed = Path("../data/processed")
print("Folder exists:", processed.exists())
print("Folder full path:", processed.resolve())
print()

if processed.exists():
    for f in sorted(processed.iterdir()):
        print(f"{f.name:40s} {f.stat().st_size/1024:10.1f} KB")

Notebook is running from: d:\project\explainable-network-threat-intelligence\notebooks

Folder exists: True
Folder full path: D:\project\explainable-network-threat-intelligence\data\processed

_debug.csv                                      0.0 KB
_debug2.csv                                     0.0 KB
live_flows.csv                                  3.6 KB
live_flows_new.csv                              3.6 KB
live_flows_v2.csv                               0.0 KB
live_traffic.pcap                              35.0 KB
preprocessed_data.pkl                       64848.6 KB
README.md                                       0.1 KB


In [16]:
import pandas as pd
pd.set_option("display.max_columns", 50)

live = pd.read_csv("../data/processed/live_flows.csv")

total_packets = live["tot_fwd_pkts"].sum() + live["tot_bwd_pkts"].sum()
print("Packets inside the flows:", total_packets)
print("Packets in the capture : 100  (IPv4 = 27, IPv6 = 73)")

cols = ["src_ip", "dst_ip", "protocol", "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts",
        "fwd_pkt_len_min", "fwd_header_len", "fwd_seg_size_min",
        "init_fwd_win_byts", "init_bwd_win_byts"]
print(live[cols].to_string())

Packets inside the flows: 24
Packets in the capture : 100  (IPv4 = 27, IPv6 = 73)
                                    src_ip                                   dst_ip  protocol  flow_duration  tot_fwd_pkts  tot_bwd_pkts  fwd_pkt_len_min  fwd_header_len  fwd_seg_size_min  init_fwd_win_byts  init_bwd_win_byts
0  2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1                2603:1063:5:110::365:7ea3         6       0.149431             2             1               75               0                 0                253                253
1  2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1                       64:ff9b::3384:c16b         6       0.269672             7             3               82               0                 0                255              16385
2   2606:4700:8ca1:d8b7:9a74:ae6:ccf5:ac50  2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1         6       0.065058             3             1               74               0                 0                 16                253
3             

In [17]:
import os
print(os.getcwd())
print(os.listdir("../data/processed"))

d:\project\explainable-network-threat-intelligence\notebooks
['live_flows.csv', 'live_flows_new.csv', 'live_flows_v2.csv', 'live_traffic.pcap', 'preprocessed_data.pkl', 'README.md', '_debug.csv', '_debug2.csv']


In [18]:
from scapy.all import rdpcap, IP, IPv6, TCP, UDP
from collections import Counter

packets = rdpcap("../data/processed/live_traffic.pcap")
print("Total packets:", len(packets))

count = Counter()
for p in packets:
    ip = "IPv4" if p.haslayer(IP) else "IPv6" if p.haslayer(IPv6) else "no-IP"
    kind = "TCP" if p.haslayer(TCP) else "UDP" if p.haslayer(UDP) else "other"
    count[(ip, kind)] += 1

for key, number in sorted(count.items()):
    print(key, number)

Total packets: 100
('IPv4', 'TCP') 4
('IPv4', 'UDP') 2
('IPv6', 'TCP') 77
('IPv6', 'UDP') 15
('no-IP', 'other') 2


In [19]:
from scapy.all import rdpcap, IP, IPv6, TCP, UDP
from collections import Counter

packets = rdpcap("../data/processed/live_traffic.pcap")

flows = Counter()
for p in packets:
    if p.haslayer(IP):
        ip, version = p[IP], "IPv4"
    elif p.haslayer(IPv6):
        ip, version = p[IPv6], "IPv6"
    else:
        continue

    if p.haslayer(TCP):
        layer, proto = p[TCP], "TCP"
    elif p.haslayer(UDP):
        layer, proto = p[UDP], "UDP"
    else:
        continue

    # Same two computers + same two ports = same conversation
    ends = sorted([(ip.src, layer.sport), (ip.dst, layer.dport)])
    flows[(version, proto, ends[0], ends[1])] += 1

print("Conversations that should exist:", len(flows))
print()
for (version, proto, a, b), n in flows.most_common():
    print(version, proto, "ports", a[1], "<->", b[1], "packets:", n)
    

Conversations that should exist: 18

IPv6 TCP ports 57029 <-> 443 packets: 29
IPv6 TCP ports 59091 <-> 443 packets: 14
IPv6 UDP ports 65464 <->

 443 packets: 11
IPv6 TCP ports 59636 <-> 443 packets: 6
IPv6 TCP ports 54622 <-> 443 packets: 6
IPv4 TCP ports 57028 <-> 443 packets: 4
IPv6 TCP ports 59829 <-> 443 packets: 3
IPv6 TCP ports 53577 <-> 443 packets: 3
IPv6 TCP ports 49960 <-> 443 packets: 3
IPv6 TCP ports 63571 <-> 443 packets: 3
IPv6 TCP ports 60183 <-> 443 packets: 2
IPv4 UDP ports 53 <-> 57574 packets: 2
IPv6 UDP ports 57574 <-> 53 packets: 2
IPv6 UDP ports 443 <-> 63677 packets: 2
IPv6 TCP ports 5228 <-> 64392 packets: 2
IPv6 TCP ports 65381 <-> 443 packets: 2
IPv6 TCP ports 64802 <-> 443 packets: 2
IPv6 TCP ports 51575 <-> 443 packets: 2


In [20]:
import os
import pandas as pd

path = "../data/processed/live_flows_new.csv"
print("File exists:", os.path.exists(path))
if os.path.exists(path):
    new = pd.read_csv(path)
    print("Flows:", len(new))
    print("Packets inside flows:", new["tot_fwd_pkts"].sum() + new["tot_bwd_pkts"].sum())

File exists: True
Flows: 6
Packets inside flows: 24


In [21]:
from scapy.all import rdpcap
from cicflowmeter.flow import Flow
from cicflowmeter.features.context import PacketDirection

packets = rdpcap("../data/processed/live_traffic.pcap")

# Test 1: which packets have no 'flags'?
bad, total = [], 0
for i, p in enumerate(packets):
    if "TCP" not in p and "UDP" not in p:
        continue
    total += 1
    try:
        p.flags
    except AttributeError:
        bad.append((i, "IPv6" if p.haslayer("IPv6") else "IPv4", "TCP" if "TCP" in p else "UDP"))
print("TCP/UDP packets:", total)
print("Packets with no 'flags':", len(bad))
print("First 5:", bad[:5])

# Test 2: does a new Flow already contain its first packet?
first = next(p for p in packets if "TCP" in p or "UDP" in p)
f = Flow(first, PacketDirection.FORWARD)
print("Packets in a brand-new Flow:", len(f.packets))
f.add_packet(first, PacketDirection.FORWARD)
print("After process() adds it again:", len(f.packets))

TCP/UDP packets: 98
Packets with no 'flags': 15
First 5: [(13, 'IPv6', 'UDP'), (20, 'IPv6', 'UDP'), (29, 'IPv6', 'UDP'), (44, 'IPv6', 'UDP'), (62, 'IPv6', 'UDP')]
Packets in a brand-new Flow: 1
After process() adds it again: 2


In [22]:
import os, traceback
from scapy.all import rdpcap
from cicflowmeter.flow_session import FlowSession

# Size of the file from the last command-line run
path = "../data/processed/live_flows_v2.csv"
print("live_flows_v2.csv size (bytes):", os.path.getsize(path) if os.path.exists(path) else "missing")

packets = rdpcap("../data/processed/live_traffic.pcap")
session = FlowSession(output_mode="csv", output="../data/processed/_debug.csv")

for p in packets:
    session.process(p)

print("Flows built:", len(session.flows))
print("Packets inside the flows:", sum(len(f.packets) for f in session.flows.values()))

# Now try to calculate the numbers for every flow and show the first error
errors = 0
for key, flow in session.flows.items():
    try:
        flow.get_data()
    except Exception:
        errors += 1
        if errors == 1:
            traceback.print_exc()
print("Flows that fail when calculating numbers:", errors)

live_flows_v2.csv size (bytes): 0
Flows built: 0
Packets inside the flows: 0
Flows that fail when calculating numbers: 0


In [23]:
import inspect, os
from scapy.all import rdpcap
from cicflowmeter import constants
from cicflowmeter.flow_session import FlowSession
from cicflowmeter.features.context import PacketDirection, get_packet_flow_key

print("Edited code is loaded:", "is_new" in inspect.getsource(FlowSession.process))
print("EXPIRED_UPDATE:", constants.EXPIRED_UPDATE, "  PACKETS_PER_GC:", constants.PACKETS_PER_GC)

packets = rdpcap("../data/processed/live_traffic.pcap")
session = FlowSession(output_mode="csv", output="../data/processed/_debug2.csv")

for i, p in enumerate(packets):
    session.process(p)
    if i in (0, 1, 2, 5, 10, 50, 97):
        print(f"after packet {i}: counted={session.packets_count}, flows={len(session.flows)}")

first = next(p for p in packets if "TCP" in p or "UDP" in p)
try:
    print("key works, ports:", get_packet_flow_key(first, PacketDirection.FORWARD)[2:])
except Exception as e:
    print("key FAILED:", type(e).__name__, e)

path = "../data/processed/_debug2.csv"
print("debug file size:", os.path.getsize(path) if os.path.exists(path) else "missing")

Edited code is loaded: False
EXPIRED_UPDATE: 240   PACKETS_PER_GC: 1000
after packet 0: counted=0, flows=0
after packet 1: counted=0, flows=0
after packet 2: counted=0, flows=0
after packet 5: counted=0, flows=0
after packet 10: counted=0, flows=0
after packet 50: counted=0, flows=0
after packet 97: counted=0, flows=0
key works, ports: (60183, 443)
debug file size: 0


In [24]:
import inspect
from cicflowmeter.flow_session import FlowSession

print("File:", inspect.getsourcefile(FlowSession))
print("Methods:", [n for n in vars(FlowSession) if not n.startswith("__")])
print("process comes from:", FlowSession.process.__qualname__)

File: C:\Users\RONAK SOLANKI\AppData\Roaming\Python\Python313\site-packages\cicflowmeter\flow_session.py
Methods: ['toPacketList', 'get_flows', 'garbage_collect', 'flush_flows']
process comes from: DefaultSession.process


In [25]:
import pandas as pd
v3 = pd.read_csv("../data/processed/live_flows_v3.csv")
print("Flows:", len(v3))
print("Packets inside flows:", v3["tot_fwd_pkts"].sum() + v3["tot_bwd_pkts"].sum())
print("Biggest flow (packets):", (v3["tot_fwd_pkts"] + v3["tot_bwd_pkts"]).max())

Flows: 18
Packets inside flows: 98
Biggest flow (packets): 29


In [26]:
from scapy.all import rdpcap, IP, IPv6, TCP, UDP
import pandas as pd

def payload_len(pkt):
    """Bytes of data inside TCP/UDP (no Ethernet, IP or TCP/UDP headers)."""
    if pkt.haslayer(TCP):
        l4_header = pkt[TCP].dataofs * 4
    elif pkt.haslayer(UDP):
        return max(pkt[UDP].len - 8, 0)
    else:
        return 0
    if pkt.haslayer(IP):
        return max(pkt[IP].len - pkt[IP].ihl * 4 - l4_header, 0)
    if pkt.haslayer(IPv6):
        return max(pkt[IPv6].plen - l4_header, 0)
    return 0

packets = rdpcap("../data/processed/live_traffic.pcap")
rows = []
for p in packets:
    if not (p.haslayer(TCP) or p.haslayer(UDP)):
        continue
    rows.append({
        "ip": "IPv4" if p.haslayer(IP) else "IPv6",
        "proto": "TCP" if p.haslayer(TCP) else "UDP",
        "whole_packet": len(p),
        "payload": payload_len(p),
    })

df_len = pd.DataFrame(rows)
summary = df_len.groupby(["ip", "proto"]).agg(
    packets=("payload", "size"),
    avg_whole_packet=("whole_packet", "mean"),
    avg_payload=("payload", "mean"),
    max_payload=("payload", "max"),
    zero_payload=("payload", lambda s: int((s == 0).sum())),
).round(1)
print(summary)
print("Total packets:", len(df_len))

            packets  avg_whole_packet  avg_payload  max_payload  zero_payload
ip   proto                                                                   
IPv4 TCP          4              66.0          0.0            0             4
     UDP          2             158.5        116.5          183             0
IPv6 TCP         77             384.7        309.6         1300            26
     UDP         15             264.9        202.9          615             0
Total packets: 98


In [27]:
import sys, importlib
sys.path.append("../src")
import flow_features
importlib.reload(flow_features)

import pandas as pd
from scapy.all import rdpcap
from cicflowmeter.flow_session import FlowSession

session = FlowSession(output_mode="csv", output="../data/processed/_debug.csv")
for p in rdpcap("../data/processed/live_traffic.pcap"):
    session.process(p)

flows = list(session.flows.values())
print("Flows:", len(flows), "  Packets:", sum(len(f.packets) for f in flows))

new = pd.DataFrame([flow_features.build_features(f.packets) for f in flows])
cols = ["Flow Duration", "Flow Bytes/s", "Total Length of Fwd Packets",
        "Total Length of Bwd Packets", "Fwd Packet Length Min", "Fwd Packet Length Max"]
print(new[cols].round(1).to_string())

total = new["Total Length of Fwd Packets"].sum() + new["Total Length of Bwd Packets"].sum()
print("Total data bytes in all flows:", total)
print("Flows with empty rates (zero duration):", int(new["Flow Bytes/s"].isna().sum()))

Flows: 0   Packets: 0


KeyError: "None of [Index(['Flow Duration', 'Flow Bytes/s', 'Total Length of Fwd Packets',\n       'Total Length of Bwd Packets', 'Fwd Packet Length Min',\n       'Fwd Packet Length Max'],\n      dtype='object')] are in the [columns]"

In [28]:
import inspect
from cicflowmeter.flow_session import FlowSession

print("process comes from:", FlowSession.process.__qualname__)
print("Edited code loaded:", "is_new" in inspect.getsource(FlowSession.process))
print("Flows after processing:", len(flows))

process comes from: DefaultSession.process
Edited code loaded: False
Flows after processing: 0


## Phase 8: Live Attack Prediction & Uncertainty Analysis

Now that feature mapping and compatibility checks are validated, we perform live model predictions, calculate prediction uncertainty, and categorize confidence levels.

In [ ]:
# Feature Cleaning & Unit Alignment (Microsecond conversion for time features)
time_features = [
    "Flow Duration", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
    "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Active Mean", "Active Std", "Active Max", "Active Min",
    "Idle Mean", "Idle Std", "Idle Max", "Idle Min"
]

X_live_clean = live_features.copy()
for col in time_features:
    if col in X_live_clean.columns and X_live_clean[col].dropna().median() < 1000:
        X_live_clean[col] = X_live_clean[col] * 1_000_000.0

X_live_clean = X_live_clean.replace([np.inf, -np.inf], 0.0).fillna(0.0).clip(lower=0.0)
print("Cleaned live features shape:", X_live_clean.shape)


In [ ]:
# Predict attacks and probabilities
predictions = model.predict(X_live_clean)
probabilities = model.predict_proba(X_live_clean)

# Calculate Confidence and Uncertainty
confidence = np.max(probabilities, axis=1)
uncertainty = 1.0 - confidence
confidence_threshold = 0.70

status = [
    "High Confidence" if c >= confidence_threshold else "Low Confidence (Manual Review)"
    for c in confidence
]

label_map = {0: "BENIGN", 1: "ATTACK"}
proto_map = {6: "TCP", 17: "UDP"}

results_df = pd.DataFrame({
    "flow_id": range(1, len(live_flows) + 1),
    "src_ip": live_flows["src_ip"] if "src_ip" in live_flows.columns else "N/A",
    "dst_ip": live_flows["dst_ip"] if "dst_ip" in live_flows.columns else "N/A",
    "protocol": [proto_map.get(p, str(p)) for p in live_flows["protocol"]],
    "prediction": predictions,
    "label": [label_map[p] for p in predictions],
    "confidence": np.round(confidence, 4),
    "uncertainty": np.round(uncertainty, 4),
    "status": status
})

display(results_df)
